<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/05_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Fine-tuning

Fine-tuning последнего блока ResNet18: сначала независимое обучение head-only модели, затем разморозка `layer4` + `fc` и разные learning rate для двух групп параметров.

> Этот notebook самодостаточен: его можно запускать сверху вниз в чистом Google Colab. Он не требует выполнения других notebook-файлов проекта.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

raw_dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

labels = np.array([raw_dataset[i][1] for i in range(len(raw_dataset))])

train_indices, val_indices = train_test_split(
    np.arange(len(raw_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

class_names = raw_dataset.classes
weights = ResNet18_Weights.IMAGENET1K_V1
resnet_transform = weights.transforms()

resnet_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_train_dataset = Subset(resnet_train_source, train_indices.tolist())
resnet_val_dataset = Subset(resnet_val_source, val_indices.tolist())

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

resnet_val_loader = DataLoader(
    resnet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

loss_fn = nn.CrossEntropyLoss()

print("Train:", len(resnet_train_dataset))
print("Validation:", len(resnet_val_dataset))
print("Classes:", len(class_names))


In [ ]:
import torch
from sklearn.metrics import f1_score


def evaluate_metrics(model, dataloader, loss_fn, device, num_classes):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_top3_correct = 0
    total_objects = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3 = logits.topk(k=3, dim=1).indices

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_top3_correct += (
                (top3 == labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )
            total_objects += batch_size

            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )

    return {
        "loss": total_loss / total_objects,
        "accuracy": total_correct / total_objects,
        "macro_f1": macro_f1,
        "top3_accuracy": total_top3_correct / total_objects,
    }

## Подготовка head-only модели

Чтобы notebook был полностью независимым, сначала здесь обучается `fc` при замороженном backbone. Полученный лучший checkpoint затем используется как старт для fine-tuning `layer4`.

In [ ]:
def train_head_one_epoch(
    model, dataloader, loss_fn, optimizer, device
):
  model.eval()
  model.fc.train()

  total_loss = 0
  total_correct = 0
  total_objects = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)
    total_loss += loss.item() * batch_size
    total_correct += (
        logits.argmax(dim=1) == labels
    ).sum().item()
    total_objects += batch_size

  return (
      total_loss / total_objects,
      total_correct / total_objects
  )

In [ ]:
seed_everything(42)
resnet_train_loader.generator.manual_seed(42)

resnet_model = resnet18(weights=weights)

for parameter in resnet_model.parameters():
  parameter.requires_grad = False

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    len(class_names),
)

resnet_model = resnet_model.to(device)

# Передаем оптимизатору только параметры последнего слоя
resnet_optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=0.001,
)
loss_fn = nn.CrossEntropyLoss()

resnet_checkpoint_path = Path(
    "checkpoints/resnet18_head.pth"
)

resnet_checkpoint_path.parent.mkdir(
    parents=True, exist_ok=True
)

resnet_history = []

best_macro_f1 = -float("inf")
best_epoch = None
epochs_without_improvement = 0

max_epochs = 15
patience = 3
min_delta = 1e-4

for epoch in range(1, max_epochs+1):
  train_loss, train_accuracy = train_head_one_epoch(
      resnet_model,
      resnet_train_loader,
      loss_fn,
      resnet_optimizer,
      device,
  )

  metrics = evaluate_metrics(
      resnet_model,
      resnet_val_loader,
      loss_fn,
      device,
      num_classes=len(class_names),
  )

  resnet_history.append({
      'epoch': epoch,
      'train_loss': train_loss,
      'train_accuracy': train_accuracy,
      'val_loss': metrics["loss"],
      'val_accuracy': metrics['accuracy'],
      'val_macro_f1': metrics['macro_f1'],
      'val_top3_accuracy': metrics['top3_accuracy'],
  })

  status = ""

  if metrics['macro_f1'] > best_macro_f1 + min_delta:
    best_macro_f1 = metrics['macro_f1']
    best_epoch = epoch
    epochs_without_improvement = 0

    torch.save({
        'model_state_dict': resnet_model.state_dict(),
        'epoch': epoch,
        'metrics': metrics,
        'class_names': class_names,
        'weights': 'IMAGENET1K_V1',
    }, resnet_checkpoint_path)

    status = 'saved'

  else:
    epochs_without_improvement += 1
    status = (
        f"No improvement: "
        f"{epochs_without_improvement}/{patience}"
    )

  print(
      f"Epoch {epoch}/{max_epochs} | "
      f"Train loss: {train_loss:.4f} | "
      f"Train accuracy: {train_accuracy:.2%} | "
      f"Val loss: {metrics['loss']:.4f} | "
      f"Val accuracy: {metrics['accuracy']:.2%} | "
      f"Val macro-F1: {metrics['macro_f1']:.4f} | "
      f"{status}"
  )

  if epochs_without_improvement >= patience:
    print("Early stopping")
    break

In [ ]:
checkpoint = torch.load(
    resnet_checkpoint_path,
    map_location=device,
    weights_only=True
)

resnet_model.load_state_dict(
    checkpoint['model_state_dict']
)

resnet_metrics = evaluate_metrics(
    resnet_model,
    resnet_val_loader,
    loss_fn,
    device,
    num_classes=len(class_names)
)

print("Loaded epoch:", checkpoint['epoch'])

for name, value in resnet_metrics.items():
  print(f"{name}: {value:.4f}")

## `fine-tuning` последнего блока `layer4`

Сейчас ResNet работает как фиксированный извлекатель признаков, теперь позволим ее наиболее высокоуровневым признакам немного адаптироваться к породам животоных. Это второй стандартный сценарий transfer learning после обучения классификатора. [PyTorch Transfer Learning Tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

При этом:
  * `layer1-layer3` остаются замороженными;
  * `layer4` и `fc` обучаются;
  * для `layer4` используем очень маленький learning rate;
  * обработку изображений пока не меняем, чтобы проверить только эффект разморозки


### **1 Новая функция обучения**

In [ ]:
def train_layer4_one_epoch(
    model, dataloader, loss_fn, optimizer, device
):
  # Замороженная часть остается в eval
  model.eval()

  # Дообучаемые части переводим в train
  model.fc.train()
  model.layer4.train()

  total_loss = 0.0
  total_correct = 0
  total_objects = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)

    total_loss += loss.item() * batch_size
    total_correct += (
        logits.argmax(dim=1) == labels
    ).sum().item()
    total_objects += batch_size

  return (
      total_loss / total_objects,
      total_correct / total_objects
  )


Теперь `BatchNorm` внутри `layer4` тоже работает в режиме обучения, а статистики замороженных блоков не обновляются

### **2 Восстанавливаем лучшую head-only модель**

In [ ]:
seed_everything(42)
resnet_train_loader.generator.manual_seed(42)

head_checkpoint = torch.load(
    resnet_checkpoint_path,
    map_location="cpu",
    weights_only=True
)

fine_tune_model = resnet18(weights=None)

fine_tune_model.fc = nn.Linear(
    fine_tune_model.fc.in_features,
    len(class_names),
)

fine_tune_model.load_state_dict(
    head_checkpoint["model_state_dict"]
)

# Сначала замораживаем всю модель
for parameter in fine_tune_model.parameters():
  parameter.requires_grad = False

# Размораживаем layer4
for parameter in fine_tune_model.layer4.parameters():
  parameter.requires_grad = True

# И классификатор
for parameter in fine_tune_model.fc.parameters():
  parameter.requires_grad = True

fine_tune_model = fine_tune_model.to(device)

Мы используем `weights=None`, потому что веса ImageNet сейчас не нужны: сразу после создания архитектуры загружается наш checkpoint 11-й эпохи

Проверим параметры:

In [ ]:
trainable_names = [
    name
    for name, parameter in fine_tune_model.named_parameters()
    if parameter.requires_grad
]

assert all(
    name.startswith(("layer4.", "fc."))
    for name in trainable_names
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in fine_tune_model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", trainable_parameters)
print("First trainable parameter:", trainable_names[0])
print("Last trainable parameter:", trainable_names[-1])

### 3 Разные learning rate

In [ ]:
fine_tune_optimizer = torch.optim.Adam(
    [
        {
            "params": fine_tune_model.layer4.parameters(),
            "lr": 1e-5,
        },
        {
            "params": fine_tune_model.fc.parameters(),
            "lr": 1e-4,
        }
    ]
)

`layer4` уже содержит полезные предобученные признаки, поэтому изменяем ее осторожнее. `fc` получает learning rate в десять раз больше. PyTorch позволяет задавать отдельные настройки для групп параметров через optimizer parameters groups

### 4 Fine-tuning


In [ ]:
fine_tune_checkpoint_path = Path(
    "checkpoints/resnet18_layer4.pth"
)

fine_tune_history = []

# Начинаем сравнение с результатом head-only модели
best_macro_f1 = head_checkpoint["metrics"]["macro_f1"]
best_epoch = 0
epochs_without_improvement = 0

max_epochs = 10
patience = 3
min_delta = 1e-4

for epoch in range(1, max_epochs+1):
  train_loss, train_accuracy = train_layer4_one_epoch(
      fine_tune_model,
      resnet_train_loader,
      loss_fn,
      fine_tune_optimizer,
      device,
  )

  metrics = evaluate_metrics(
      fine_tune_model,
      resnet_val_loader,
      loss_fn,
      device,
      num_classes=len(class_names),
  )

  fine_tune_history.append({
      "epoch": epoch,
      "train_loss": train_loss,
      "train_accuracy": train_accuracy,
      "val_loss": metrics["loss"],
      "val_accuracy": metrics["accuracy"],
      "val_macro_f1": metrics["macro_f1"],
      "val_top3_accuracy": metrics["top3_accuracy"],
  })

  if metrics['macro_f1'] > best_macro_f1 + min_delta:
    best_macro_f1 = metrics['macro_f1']
    best_epoch = epoch
    epochs_without_improvement = 0
    status = 'saved'

    torch.save(
        {
            "model_state_dict": fine_tune_model.state_dict(),
            "optimizer_state_dict": fine_tune_optimizer.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "class_names": class_names,
            "weights": "IMAGENET1K_V1",
            "stage": "layer4_fine_tuning",
        },
        fine_tune_checkpoint_path
    )

  else:
      epochs_without_improvement += 1
      status = (
          f"No improvement: "
          f"{epochs_without_improvement}/{patience}"
      )

  print(
      f"Epoch {epoch}/{max_epochs} | "
      f"Train loss: {train_loss:.4f} | "
      f"Train accuracy: {train_accuracy:.2%} | "
      f"Val loss: {metrics['loss']:.4f} | "
      f"Val accuracy: {metrics['accuracy']:.2%} | "
      f"Val macro-F1: {metrics['macro_f1']:.4f} | "
      f"{status}"
  )

  if epochs_without_improvement >= patience:
      print("Early stopping")
      break
